In [1]:
import pandas as pd
import random
from transformers import pipeline
from pathlib import Path

c:\Coding\pba\tugas-klp\LPDP-SentimentAnalysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")
print("Generator siap!")

Generator siap!


c:\Coding\pba\tugas-klp\LPDP-SentimentAnalysis\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [3]:
BASE_DIR = Path(r"c:\Coding\pba\tugas-klp\LPDP-SentimentAnalysis")

TRAIN_A_PATH = BASE_DIR / "outputs/output_split/track_a_train.csv"
TRAIN_B_PATH = BASE_DIR / "outputs/output_split/track_b_train.csv"

LABEL_COL  = "Sentiment"
TEXT_COL_A = "text_clean"
TEXT_COL_B = "text_bert"

In [4]:
train_df_a = pd.read_csv(TRAIN_A_PATH)
train_df_b = pd.read_csv(TRAIN_B_PATH)

print("── Track A ──")
print(f"Shape : {train_df_a.shape}")
print(f"Kolom : {train_df_a.columns.tolist()}")
print(train_df_a[LABEL_COL].value_counts())

print("\n── Track B ──")
print(f"Shape : {train_df_b.shape}")
print(f"Kolom : {train_df_b.columns.tolist()}")
print(train_df_b[LABEL_COL].value_counts())

── Track A ──
Shape : (830, 4)
Kolom : ['doc_id', 'text_clean', 'Sentiment', 'label_encoded']
Sentiment
positive    308
neutral     273
negative    249
Name: count, dtype: int64

── Track B ──
Shape : (830, 4)
Kolom : ['doc_id', 'text_bert', 'Sentiment', 'label_encoded']
Sentiment
positive    308
neutral     273
negative    249
Name: count, dtype: int64


In [5]:
def get_class_distribution(df, label_col):
    """Menghitung jumlah sampel per kelas."""
    return df[label_col].value_counts().to_dict()


def compute_augmentation_targets(dist):
    """
    Menghitung berapa sampel tambahan yang dibutuhkan
    agar setiap kelas mencapai jumlah kelas mayoritas.
    """
    max_count = max(dist.values())
    return {
        label: max_count - count
        for label, count in dist.items()
        if max_count - count > 0
    }


def generate_augmented_text(base_text, label, generator, max_new_tokens=128):
    """
    Membuat variasi kalimat dari base_text menggunakan Flan-T5,
    dengan tetap mempertahankan sentimen/kategori aslinya.
    """
    prompt = (
        f"Paraphrase the following {label} sentiment text in Indonesian "
        f"so it has different sentence structure but same meaning.\n"
        f"Text: {base_text}\nParaphrase:"
    )
    result = generator(
        prompt,
        max_new_tokens = max_new_tokens,
        do_sample      = True,
        temperature    = 0.8
    )
    return result[0]["generated_text"].strip()


def auto_augment_to_balance(
    train_df,
    label_col      = LABEL_COL,
    text_col       = TEXT_COL_A,
    generator      = generator,
    max_new_tokens = 128,
    random_seed    = 42
):
    """
    Secara otomatis mengaugmentasi kelas-kelas minority
    hingga jumlahnya sama dengan kelas mayoritas.

    Returns:
        DataFrame gabungan data asli + data augmentasi.
    """
    random.seed(random_seed)

    dist    = get_class_distribution(train_df, label_col)
    targets = compute_augmentation_targets(dist)

    print("Distribusi awal:")
    for label, count in sorted(dist.items()):
        print(f"   {label}: {count}")

    print(f"\nKelas yang perlu diaugmentasi: {list(targets.keys())}")

    augmented_rows = []

    for label, n_needed in targets.items():
        print(f"\n[Augmenting] '{label}' -> perlu +{n_needed} sampel")

        pool = (
            train_df[train_df[label_col] == label][text_col]
            .dropna()
            .tolist()
        )

        for i in range(n_needed):
            base_text = random.choice(pool)

            try:
                new_text = generate_augmented_text(
                    base_text, label, generator, max_new_tokens
                )
            except Exception as e:
                print(f"  [Warning] Gagal generate sampel ke-{i+1}: {e}")
                new_text = base_text  # fallback ke teks asli

            augmented_rows.append({
                label_col:             label,
                text_col:              new_text,
                "is_augmented":        True,
                "augmentation_source": "flan-t5-paraphrase"
            })

            if (i + 1) % 10 == 0:
                print(f"  -> {i + 1}/{n_needed} sampel selesai")

    aug_df        = pd.DataFrame(augmented_rows)
    train_df_orig = train_df.copy()
    train_df_orig["is_augmented"]        = False
    train_df_orig["augmentation_source"] = "original"

    balanced_df = pd.concat([train_df_orig, aug_df], ignore_index=True)
    balanced_df = balanced_df.sample(
        frac=1, random_state=random_seed
    ).reset_index(drop=True)

    print("\n── Distribusi setelah augmentasi ──")
    print(balanced_df[label_col].value_counts().sort_index())

    return balanced_df

In [6]:
print("=" * 50)
print("AUGMENTASI TRACK A (text_clean)")
print("=" * 50)

balanced_train_df_a = auto_augment_to_balance(
    train_df       = train_df_a,
    label_col      = LABEL_COL,
    text_col       = TEXT_COL_A,
    generator      = generator,
    max_new_tokens = 128,
    random_seed    = 42
)

balanced_train_df_a.to_csv("train_balanced_flant5_track_a.csv", index=False)
print("\nTrack A disimpan ke: train_balanced_flant5_track_a.csv")
print(f"Total baris        : {len(balanced_train_df_a)}")

Token indices sequence length is longer than the specified maximum sequence length for this model (653 > 512). Running this sequence through the model will result in indexing errors


AUGMENTASI TRACK A (text_clean)
Distribusi awal:
   negative: 249
   neutral: 273
   positive: 308

Kelas yang perlu diaugmentasi: ['neutral', 'negative']

[Augmenting] 'neutral' -> perlu +35 sampel
  -> 10/35 sampel selesai
  -> 20/35 sampel selesai
  -> 30/35 sampel selesai

[Augmenting] 'negative' -> perlu +59 sampel
  -> 10/59 sampel selesai
  -> 20/59 sampel selesai
  -> 30/59 sampel selesai
  -> 40/59 sampel selesai
  -> 50/59 sampel selesai

── Distribusi setelah augmentasi ──
Sentiment
negative    308
neutral     308
positive    308
Name: count, dtype: int64

Track A disimpan ke: train_balanced_flant5_track_a.csv
Total baris        : 924


In [7]:
print("\n" + "=" * 50)
print("AUGMENTASI TRACK B (text_bert)")
print("=" * 50)

balanced_train_df_b = auto_augment_to_balance(
    train_df       = train_df_b,
    label_col      = LABEL_COL,
    text_col       = TEXT_COL_B,
    generator      = generator,
    max_new_tokens = 128,
    random_seed    = 42
)

balanced_train_df_b.to_csv("train_balanced_flant5_track_b.csv", index=False)
print("\nTrack B disimpan ke: train_balanced_flant5_track_b.csv")
print(f"Total baris        : {len(balanced_train_df_b)}")


AUGMENTASI TRACK B (text_bert)
Distribusi awal:
   negative: 249
   neutral: 273
   positive: 308

Kelas yang perlu diaugmentasi: ['neutral', 'negative']

[Augmenting] 'neutral' -> perlu +35 sampel


  -> 10/35 sampel selesai
  -> 20/35 sampel selesai
  -> 30/35 sampel selesai

[Augmenting] 'negative' -> perlu +59 sampel
  -> 10/59 sampel selesai
  -> 20/59 sampel selesai
  -> 30/59 sampel selesai
  -> 40/59 sampel selesai
  -> 50/59 sampel selesai

── Distribusi setelah augmentasi ──
Sentiment
negative    308
neutral     308
positive    308
Name: count, dtype: int64

Track B disimpan ke: train_balanced_flant5_track_b.csv
Total baris        : 924


In [8]:
print("\n" + "=" * 50)
print("RINGKASAN HASIL AUGMENTASI")
print("=" * 50)

print("\nTrack A (text_clean):")
print(balanced_train_df_a[LABEL_COL].value_counts().sort_index())
print(f"Total: {len(balanced_train_df_a)} baris")

print("\nTrack B (text_bert):")
print(balanced_train_df_b[LABEL_COL].value_counts().sort_index())
print(f"Total: {len(balanced_train_df_b)} baris")

print("\nFile output:")
print("  - train_balanced_flant5_track_a.csv")
print("  - train_balanced_flant5_track_b.csv")
print("\nAugmentasi selesai!")


RINGKASAN HASIL AUGMENTASI

Track A (text_clean):
Sentiment
negative    308
neutral     308
positive    308
Name: count, dtype: int64
Total: 924 baris

Track B (text_bert):
Sentiment
negative    308
neutral     308
positive    308
Name: count, dtype: int64
Total: 924 baris

File output:
  - train_balanced_flant5_track_a.csv
  - train_balanced_flant5_track_b.csv

Augmentasi selesai!
